# PoC Prophet — pronóstico y estacionalidad de precio (1 producto × 1 mercado)

**Issue #27** · `area:ml` · Milestone M4

Objetivo: validar si Prophet aporta sobre el baseline en una serie de precio de
SISAP y **confirmar el umbral de historia** (`config.MIN_OBSERVACIONES` /
`MIN_SPAN_DIAS`) antes de industrializar el entrenamiento (ml-3). Decisión
go/no-go al final.

Requiere `pip install -e ".[ml,notebooks]"` y `SUPABASE_DB_URL` en el entorno
(mismo connection string del pooler que usa la capa de carga).

## 0. Setup y parámetros

In [ ]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from prophet import Prophet

from observatorio.ml import config
from observatorio.ml.baseline import pronostico_media_movil, pronostico_naive
from observatorio.ml.datos import Serie, cargar_series

load_dotenv()  # SUPABASE_DB_URL desde .env si existe
warnings.filterwarnings("ignore")  # Prophet/cmdstanpy son ruidosos
pd.set_option("display.max_rows", 50)

# Mercado a explorar en la PoC (una sola fuente para mantenerlo simple).
FUENTE = "sisap_minorista"
HORIZONTE = config.HORIZONTE_DIAS

## 1. Cargar series desde `gold.fct_precio_diario`

`cargar_series` devuelve una `Serie` por `(fuente, cod_departamento, producto)`.
Listamos las de mayor historia para elegir la de la PoC.

In [ ]:
series = cargar_series(fuentes=[FUENTE])

resumen = (
    pd.DataFrame(
        {
            "producto": [s.producto for s in series],
            "n_obs": [s.n_obs for s in series],
            "span_dias": [s.span_dias for s in series],
            "modelable": [s.es_modelable() for s in series],
        }
    )
    .sort_values("n_obs", ascending=False)
    .reset_index(drop=True)
)
print(f"{len(series)} series en {FUENTE}")
resumen.head(15)

In [ ]:
# Elegimos la serie con más historia (la mejor candidata para Prophet).
serie = max(series, key=lambda s: (s.n_obs, s.span_dias))
print(f"PoC sobre: {serie.producto}  ·  n_obs={serie.n_obs}  ·  span={serie.span_dias}d")
assert serie.es_modelable(), "La mejor serie no supera el umbral de historia"
serie.obs.tail()

## 2. EDA — la serie observada

Una fila por día observado (SISAP publica interdiario y el backfill va muestreado,
así que la serie es esparsa: lo veremos en los saltos entre puntos).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(serie.obs["ds"], serie.obs["y"], marker=".", linewidth=0.8)
ax.set_title(f"{FUENTE} · {serie.producto} — precio observado")
ax.set_xlabel("fecha"); ax.set_ylabel("precio (S/)")
plt.tight_layout(); plt.show()

serie.obs["y"].describe()

## 3. Ajustar Prophet y pronosticar

Desactivamos estacionalidad anual (no hay años de historia propia todavía) y
diaria (el grano es diario). Dejamos la semanal, la única con soporte muestral.

In [ ]:
m = Prophet(
    weekly_seasonality=True,
    yearly_seasonality=False,
    daily_seasonality=False,
)
m.fit(serie.obs)  # obs ya viene con columnas ds / y

future = m.make_future_dataframe(periods=HORIZONTE)
forecast = m.predict(future)
forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(HORIZONTE)

In [ ]:
fig = m.plot(forecast)
plt.title(f"{serie.producto} — Prophet (bandas de incertidumbre)")
plt.tight_layout(); plt.show()

### 3.1 Componentes

Tendencia + estacionalidad semanal. ¿Hay un patrón de día de semana
interpretable (p.ej. precios de fin de semana distintos)?

In [ ]:
fig = m.plot_components(forecast)
plt.tight_layout(); plt.show()

## 4. ¿Le gana al baseline? (holdout walk-forward simple)

Reservamos los últimos `HORIZONTE` días observados como test, reentrenamos con el
resto y comparamos el MAPE de Prophet contra el naive y la media móvil. Es una
validación rápida de PoC; el backtesting formal va en ml-4.

**Caveat de series esparsas:** las fechas de test no son necesariamente los N días
de calendario siguientes, así que alineamos el pronóstico a las fechas realmente
observadas por fecha (`merge` en `ds`).

In [ ]:
def mape(real: pd.Series, pred: pd.Series) -> float:
    return float((abs(real.values - pred.values) / real.values).mean() * 100)

corte = serie.n_obs - HORIZONTE
train = Serie(serie.fuente, serie.cod_departamento, serie.producto,
              serie.obs.iloc[:corte].reset_index(drop=True))
test = serie.obs.iloc[corte:].reset_index(drop=True)
print(f"train={train.n_obs} obs · test={len(test)} obs")

In [ ]:
# Prophet entrenado solo con train, alineado a las fechas de test.
mp = Prophet(weekly_seasonality=True, yearly_seasonality=False, daily_seasonality=False)
mp.fit(train.obs)
fc = mp.predict(mp.make_future_dataframe(periods=HORIZONTE + 30))
pred_prophet = test.merge(fc[["ds", "yhat"]], on="ds", how="left").dropna(subset=["yhat"])

# Baselines: proyectan un valor constante desde el fin de train; alineamos por fecha.
pred_naive = test.merge(
    pronostico_naive(train, horizonte=HORIZONTE + 30), on="ds", how="inner",
)
pred_mm = test.merge(
    pronostico_media_movil(train, horizonte=HORIZONTE + 30), on="ds", how="inner",
)

print("MAPE (%) sobre las fechas de test observadas:")
if len(pred_prophet):
    print(f"  Prophet     : {mape(pred_prophet['y'], pred_prophet['yhat']):.2f}")
if len(pred_naive):
    print(f"  naive       : {mape(pred_naive['y'], pred_naive['yhat']):.2f}")
if len(pred_mm):
    print(f"  media_movil : {mape(pred_mm['y'], pred_mm['yhat']):.2f}")

## 5. Hallazgos y decisión go/no-go

_Completar tras correr el notebook con datos reales:_

- **Historia disponible.** ¿Cuántas series de SISAP superan el umbral
  (`modelable`)? ¿El umbral 60 obs / 90 días es razonable o hay que ajustarlo en
  `config`?
- **Estacionalidad.** ¿La componente semanal es interpretable o ruido? Si es
  ruido, desactivarla en ml-3.
- **Prophet vs baseline.** ¿Prophet le gana al naive en MAPE? Si no le gana de
  forma consistente, en ml-3 Prophet queda solo para las series con historia
  holgada y el resto se sirve con baseline.
- **Esparsidad.** ¿Cuántos puntos de test cayeron sobre fechas pronosticadas?
  Esto define cómo modelar el backtesting formal en ml-4.

**Decisión:** _go / no-go + parámetros confirmados._